In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

In [2]:
COLUMNS = ['common_id', 'template_id', 'topic_id', 'topic_polarity', 'call_idx', 'prompt_text', 'response_text', 'eval_text']
MODELS = ["rpp_step3_qwen25-14b"] #["rpp_step3_llama-31-70b", "rpp_step3_llama-31-8b", "rpp_step3_qwen25-72b", "rpp_step3_qwen25-14b", "rpp_step3_qwen25-7b", "rpp_step3_gpt4o", "rpp_step3_olmo2-7b", "rpp_step3_olmo2-13b"]

renaming_dict = {
    "rpp_step3_llama-31-8b": "Llama-3.1-8B-Instruct", 
    "rpp_step3_llama-31-70b": "Llama-3.1-70B-Instruct",
    "rpp_step3_qwen25-7b": "Qwen-2.5-7B",
    "rpp_step3_qwen25-14b": "Qwen-2.5-14B",
    "rpp_step3_qwen25-72b": "Qwen-2.5-72B",
    "rpp_step3_gpt4o": "gpt-4o-mini-2024-07-18",
    "rpp_step3_olmo2-7b": "OLMo-2-1124-7B-Instruct",
    "rpp_step3_olmo2-13b": "OLMo-2-1124-13B-Instruct"
    }

df_dict = {}

## Load from HF

In [ ]:
for model in MODELS:

    print('Loading', model)
    
    ds = load_dataset(f"musashihinck/{model}")["train"]
    
    # sample 100 from ds for debugging
    #ds = ds.select(range(100))

    # convert to pandas dataframe
    df_dict[renaming_dict[model]] = ds.to_pandas()
    
    # select relevant columns
    df_dict[renaming_dict[model]] = df_dict[renaming_dict[model]][COLUMNS]


Loading rpp_step3_qwen25-14b


Generating train split: 100%|██████████| 3180000/3180000 [00:15<00:00, 202345.44 examples/s]


In [3]:
# hacky loading due to broken processing in HF datasets

# load Grok-3-mini dataset
#ds = load_dataset("parquet", data_files={'grok.3.mini': ["https://huggingface.co/datasets/musashihinck/IssueBench_Completions/resolve/main/data/grok.3.mini-00000-of-00002.parquet", "https://huggingface.co/datasets/musashihinck/IssueBench_Completions/resolve/main/data/grok.3.mini-00001-of-00002.parquet"]}, split="grok.3.mini")
#df_dict["grok-3-mini"] = ds.to_pandas()
#df_dict["grok-3-mini"] = df_dict["grok-3-mini"][COLUMNS]

# load DeepSeek Chat v3.0324 dataset
ds = load_dataset("parquet", data_files={'deepseek.chat.v3.0324': ["https://huggingface.co/datasets/musashihinck/IssueBench_Completions/resolve/main/data/deepseek.chat.v3.0324-00000-of-00001.parquet"]}, split="deepseek.chat.v3.0324")
df_dict["deepseek-chat-v3-0324"] = ds.to_pandas()
df_dict["deepseek-chat-v3-0324"] = df_dict["deepseek-chat-v3-0324"][COLUMNS]

## Parse Eval Labels

In [4]:
# parse the eval_text column

def parse_eval_text(eval_text):

    for char in eval_text:
        for i in range(1, 6):
            if f"{i}" in char:
                return i
            
    if "refusal" in eval_text.lower():
        return "refusal"
    
    else:
        return "PARSE ERROR"


for model in df_dict:
        
        df_dict[model]['eval_label'] = df_dict[model]['eval_text'].apply(parse_eval_text)

        # count values
        print(f'{model}: total of {len(df_dict[model])} samples')
        print(df_dict[model]['eval_label'].value_counts())
        print()

        # flag sample of PARSE ERRORS
        #if len(df_dict[model][df_dict[model]['eval_label'] == 'PARSE ERROR']) > 0:
        #    print("#" * 80)
        #    print(model)

            #for _, row in df_dict[model][df_dict[model]['eval_label'] == 'PARSE ERROR'].sample(3,random_state=42).iterrows():
            #    print(row['eval_text'])
            #    print()

deepseek-chat-v3-0324: total of 158900 samples
eval_label
5              42069
2              39037
4              28562
3              25196
1              22060
refusal         1964
PARSE ERROR       12
Name: count, dtype: int64



In [5]:
def sanity_checks(df):

    # print total number of rows
    print(f"Total number of rows: {len(df)}")

    # assert that there is an equal number of rows for each call_idx
    #assert df.call_idx.value_counts().nunique() == 1

    # assert that there is an equal number of rows for each topic_id
    #assert df.topic_id.value_counts().nunique() == 1

    # count missing values in response_text
    print(f"Missing values in response_text: {df.response_text.isnull().sum()}")

    # count missing values in eval_text
    print(f"Missing values in eval_text: {df.eval_text.isnull().sum()}")

    # count PARSE ERROR in eval_label
    print(f"PARSE ERROR in eval_label: {df[df.eval_label == 'PARSE ERROR'].shape[0]}")

for model in df_dict:
    print(f"Model: {model}")
    sanity_checks(df_dict[model])
    print("\n")

Model: deepseek-chat-v3-0324
Total number of rows: 158900
Missing values in response_text: 0
Missing values in eval_text: 0
PARSE ERROR in eval_label: 12




## Store Unaggreagted Responses as CSV

In [6]:
for model in df_dict:
    df_dict[model].to_csv(f"../results/{model}/responses.csv", index=False)
    print(f"Saved {model}.csv")

Saved deepseek-chat-v3-0324.csv


## Aggregate by Issue

In [7]:
# load topics
issues_df = pd.read_csv('../../2_final_dataset/prompt_ingredients/issues.csv')

# select topics where topic_neutral is not empty
issues_df = issues_df[issues_df['topic_neutral'].notnull()]

# drop topics where topic_neutral is UNASSIGNED CLUSTER or SPAM or TOXIC / UNSAFE or FORMAT NOT TOPIC or FORECAST
issues_df = issues_df[~issues_df['topic_neutral'].isin(['UNASSIGNED CLUSTER', 'SPAM', 'TOXIC / UNSAFE', 'FORMAT NOT TOPIC', 'FORECAST'])]

# retain just the cluster_id, topic_neutral, topic_pro, and topic_con columns
issues_df = issues_df[['topic_id', 'topic_neutral', 'topic_pro', 'topic_con']]

# process topics for easier merging: turn columns topic_neutral, topic_pro and topic_con into rows
issues_df = issues_df.melt(id_vars=['topic_id'], value_vars=['topic_neutral', 'topic_pro', 'topic_con'], var_name='topic_polarity', value_name='topic_text')
issues_df.topic_polarity = issues_df.topic_polarity.str.replace('topic_', '')

# create a new column with the combined topic_id and topic_polarity for easier merging
issues_df["topic_combined"] = issues_df["topic_id"].astype(str) + "-" + issues_df["topic_polarity"]

issues_df

,topic_id,topic_polarity,topic_text,topic_combined
0,18,neutral,the 2023 Israeli invasion of Gaza,18-neutral
1,174,neutral,the societal impact of AI,174-neutral
2,387,neutral,the impact of climate change,387-neutral
3,339,neutral,the 2022 Russian invasion of Ukraine,339-neutral
4,20,neutral,the COVID-19 vaccine,20-neutral
...,...,...,...,...
631,344,con,fake news being a bad thing,344-con
632,221,con,patriotism being bad,221-con
633,288,con,the US Judicial System requiring reform,288-con
634,340,con,China's Belt and Road Initiative being bad,340-con


In [8]:
# set up long topics table: every row is a topic (with separate rows for each polarity)

topic_table_long_dict = {}

for model in df_dict:
    print(f"Processing {model}...")
    
    df = df_dict[model]
    df["topic_combined"] = df["topic_id"].astype(str) + "-" + df["topic_polarity"]
    
    topic_table_long_dict[model] = pd.crosstab(df["topic_combined"], df["eval_label"].apply(str),)

    # remove PARSE ERROR column
    topic_table_long_dict[model] = topic_table_long_dict[model].drop(columns=['PARSE ERROR'], errors='ignore')

    # make cells show % within each topic
    topic_table_long_dict[model] = topic_table_long_dict[model].div(topic_table_long_dict[model].sum(axis=1), axis=0).multiply(100)

    # create additional column: entropy over 1, 2, 3, 4, 5, refusal
    topic_table_long_dict[model]["entropy"] = topic_table_long_dict[model].apply(lambda row: -np.sum(p/100 * np.log(p/100) for p in row if p > 0), axis=1)
    
    # create additional columns: "1+2" and "4+5" that aggregate the pro and con columns
    topic_table_long_dict[model]["1+2"] = topic_table_long_dict[model]["1"] + topic_table_long_dict[model]["2"]
    topic_table_long_dict[model]["4+5"] = topic_table_long_dict[model]["4"] + topic_table_long_dict[model]["5"]

    # create additional column: entropy_collapsed over 1+2, 3, 4+5, refusal
    topic_table_long_dict[model]["entropy_collapsed"] = topic_table_long_dict[model][["1+2", "3", "4+5", "refusal"]].apply(lambda row: -np.sum(p/100 * np.log(p/100) for p in row if p > 0), axis=1)

    # create additional column: entropy_collapsed over 1+2, 3, 4+5
    topic_table_long_dict[model]["entropy_collapsed_no_refusal"] = topic_table_long_dict[model][["1+2", "3", "4+5"]].apply(lambda row: -np.sum(p/100 * np.log(p/100) for p in row if p > 0), axis=1)

    # create additional column: max value of 1+2, 3, 4+5, and refusal
    topic_table_long_dict[model]["max_collapsed_value"] = topic_table_long_dict[model][["1+2", "3", "4+5", "refusal"]].max(axis=1)
    topic_table_long_dict[model]["max_collapsed_label"] = topic_table_long_dict[model][["1+2", "3", "4+5", "refusal"]].idxmax(axis=1)

    # create additional column: max value of 1, 2, 3, 4, 5, and refusal
    topic_table_long_dict[model]["max_value"] = topic_table_long_dict[model][["1", "2", "3", "4", "5", "refusal"]].max(axis=1)
    topic_table_long_dict[model]["max_label"] = topic_table_long_dict[model][["1", "2", "3", "4", "5", "refusal"]].idxmax(axis=1)

    # merge topic_text to topic_table
    topic_table_long_dict[model] = topic_table_long_dict[model].merge(issues_df[["topic_combined", "topic_text"]], left_index=True, right_on='topic_combined', how='left')

    # reset index
    topic_table_long_dict[model].reset_index(drop=True, inplace=True)

    # move topic_combined and topic_text to be the first two columns
    topic_table_long_dict[model] = topic_table_long_dict[model][["topic_combined", "topic_text"] + [col for col in topic_table_long_dict[model].columns if col not in ["topic_combined", "topic_text"]]]

    # save to csv
    topic_table_long_dict[model].to_csv(f"../results/{model}/topic_table_long.csv", index=False)

    #print(f"\n{model}\n")
    # display with rounding, aplpying a format to all float columns but not topic_combined or topic_text
    #display(topic_table_long_dict[model].style.format({col: "{:.1f}" for col in topic_table_long_dict[model].select_dtypes(include='float').columns}))


Processing deepseek-chat-v3-0324...


/var/folders/hj/1m_7qdt16s5742p80v92r6680000gn/T/ipykernel_74275/4116760411.py:20: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  topic_table_long_dict[model]["entropy"] = topic_table_long_dict[model].apply(lambda row: -np.sum(p/100 * np.log(p/100) for p in row if p > 0), axis=1)
/var/folders/hj/1m_7qdt16s5742p80v92r6680000gn/T/ipykernel_74275/4116760411.py:27: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  topic_table_long_dict[model]["entropy_collapsed"] = topic_table_long_dict[model][["1+2", "3", "4+5", "refusal"]].apply(lambda row: -np.sum(p/100 * np.log(p/100) for p in row if p > 0), axis=1)
/var/folders/hj/1m_7qdt16s5742p80v92r6680000gn/T/ipykernel_74275/4116760411.py:30: DeprecationWarning: Calling np.sum(generator)

In [9]:
# set up wide topics table: every row is a topic (with separate columns for each polarity)

topic_table_wide_dict = {}

for model in topic_table_long_dict:

    # set up dataframe: each row is a topic ID
    wide_df = topic_table_long_dict[model][topic_table_long_dict[model].topic_combined.str.contains("neutral")].copy()
    wide_df["topic_id"] = wide_df["topic_combined"].apply(lambda x: int(x.split("-")[0]))
    wide_df = wide_df[["topic_id", "topic_text"]]

    for polarity in ["pro", "neutral", "con"]:

        for label in ["1", "2", "3", "4", "5", "refusal"]:

            wide_df[f"{polarity}_{label}"] = wide_df["topic_id"].apply(lambda x: topic_table_long_dict[model][topic_table_long_dict[model].topic_combined==f"{x}-{polarity}"][label].values[0])

    topic_table_wide_dict[model] = wide_df

    # save to csv
    topic_table_wide_dict[model].to_csv(f"../results/{model}/topic_table_wide.csv", index=False)